# Hugging Face Audio Course — Unit 5: Automatic speech recognition

Turning speech into text: which pre-trained model to reach for, which dataset to train on, how to
*measure* a transcription, and what fine-tuning actually feeds the model.

Unit 3 covered the mechanics (CTC blank collapse, Whisper's task tokens). This unit is about
**choosing, measuring and adapting**.

| # | Concept | Model / data |
|---|---------|--------------|
| 1 | CTC vs seq2seq on one clip | `wav2vec2-base-960h` vs `whisper-tiny`/`base` |
| 2 | The Whisper family, size vs speed (RTFx) | tiny / base / small |
| 3 | Transcribe vs translate | MINDS-14 `de-DE` |
| 4 | The 30-second wall, chunking, timestamps | `whisper-base` |
| 5 | Choosing a dataset | the eight English ASR corpora |
| 6 | WER by hand: S, I, D | pen and paper, then `jiwer` |
| 7 | Orthographic vs normalised WER, CER | 3 models × 8 clips |
| 8 | What the data collator builds | `whisper-tiny` |

> First run downloads up to ~1.77 GB of models to `~/.cache/huggingface` (not the repo), but if you
> ran Units 2 and 3 they are already cached. Everything runs on CPU. The full notebook takes a few
> minutes.

In [ ]:
%matplotlib inline
import warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
from datasets import load_dataset
from transformers import pipeline

SAMPLING_RATE = 16_000
CTC_MODEL     = "facebook/wav2vec2-base-960h"
WHISPER_TINY  = "openai/whisper-tiny"
WHISPER_BASE  = "openai/whisper-base"
WHISPER_SMALL = "openai/whisper-small"
# Whisper decodes with 5 beams by default: ~5x slower on CPU for a marginally better transcript.
GEN = {"task": "transcribe", "language": "english", "num_beams": 1}

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
idx = next(i for i, t in enumerate(ds["text"]) if "CHRISTMAS" in t)
clip = ds[idx]["audio"]["array"]
reference = ds[idx]["text"]

_pipes = {}
def asr(model_id):
    if model_id not in _pipes:
        _pipes[model_id] = pipeline("automatic-speech-recognition", model=model_id, device=-1)
    return _pipes[model_id]

def transcribe(model_id, array=None, **kw):
    array = clip if array is None else array
    if "whisper" in model_id:
        kw.setdefault("generate_kwargs", GEN)
    t0 = time.perf_counter()
    out = asr(model_id)({"array": np.asarray(array), "sampling_rate": SAMPLING_RATE}, **kw)
    return out, time.perf_counter() - t0

print(f"clip {idx}: {len(clip)/SAMPLING_RATE:.1f}s")
print(reference)
ipd.Audio(clip, rate=SAMPLING_RATE)

## 1. CTC vs sequence-to-sequence

**CTC** models (Wav2Vec2, HuBERT, XLSR) are encoder-only. They classify every ~20 ms frame
independently into a character, then collapse repeats and blanks. Nothing in that pipeline knows what
a *word* is, so the mistakes are **phonetic** — it spells what it heard.

**Seq2seq** models (Whisper) add a decoder that writes text token by token, conditioned on the audio
*and* on what it has already written. That decoder is a language model trained on 680,000 hours of
weakly-labelled audio, so it fixes spelling, adds punctuation and restores casing for free — but it
can also hallucinate a *fluent wrong word*, which a CTC model never does.

In [ ]:
for model_id in (CTC_MODEL, WHISPER_TINY, WHISPER_BASE):
    out, secs = transcribe(model_id)
    kind = "CTC    " if "wav2vec2" in model_id else "seq2seq"
    print(f"[{kind}] {model_id}  ({secs:.1f}s)\n   {out['text'].strip()}\n")
print("reference:\n  ", reference)

Look at *what* is wrong, not just how much. Wav2Vec2 hears the sounds right and spells them as best
it can. Whisper returns cased, punctuated English — and where it errs, it errs fluently.

## 2. The Whisper checkpoint family

Five sizes, each with an English-only `.en` twin except `large`. The `.en` checkpoints refuse a
`language`/`task` argument — there is no language to choose.

In [ ]:
family = [("tiny", 39), ("base", 74), ("small", 244), ("medium", 769), ("large", 1550)]
print("size      params   English-only twin")
for name, params in family:
    print(f"{name:<8} {params:>5} M   " + (f"openai/whisper-{name}.en" if name != "large" else "(none)"))

print("\nMeasured here (CPU, greedy):")
rows = []
for model_id in (WHISPER_TINY, WHISPER_BASE, WHISPER_SMALL):
    out, secs = transcribe(model_id)
    p = asr(model_id).model.num_parameters() / 1e6
    rtfx = (len(clip) / SAMPLING_RATE) / secs
    rows.append((model_id.split("/")[-1], p, rtfx))
    print(f"  {model_id:<22} {p:>6.0f} M   {secs:>5.1f}s   RTFx {rtfx:>5.1f}x")

**RTFx** (inverse real-time factor) = audio duration / processing time. Above 1.0 is faster than real
time. It is the number you quote next to WER when someone asks whether a model is *deployable*.

## 3. One model, two tasks: transcribe and translate

Whisper's decoder is prompted with tokens saying which language it hears and which job to do. Flip
`transcribe` to `translate` and the **same weights** emit English instead. Translation always targets
English — there is no token for any other output language.

In [ ]:
from datasets import Audio

def load_minds14(name="en-AU", split="train"):
    kwargs = dict(path="PolyAI/minds14", name=name, split=split)
    try:
        return load_dataset(**kwargs)
    except Exception:
        return load_dataset(**kwargs, trust_remote_code=True)

de = load_minds14("de-DE").cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
row = de[0]
for task in ("transcribe", "translate"):
    out, _ = transcribe(WHISPER_SMALL, row["audio"]["array"],
                        generate_kwargs={"task": task, "language": "german", "num_beams": 1})
    print(f"{task:<10}: {out['text'].strip()}")
print(f"\nreference (de): {row['transcription']}")
print(f"gold English  : {row['english_transcription']}")
ipd.Audio(row["audio"]["array"], rate=SAMPLING_RATE)

## 4. Audio longer than 30 seconds

Whisper's encoder input is fixed at 80 mel bins × 3000 frames = **exactly 30 seconds**. Longer audio
must be split. There are two ways, and one trap.

In [ ]:
parts, total = [], 0.0
for i in range(len(ds)):
    a = ds[i]["audio"]["array"]; parts.append(a); total += len(a) / SAMPLING_RATE
    if total > 36: break
long_clip = np.concatenate(parts)
print(f"{total:.1f}s clip from {len(parts)} utterances\n")

# (a) the trap: no chunking and no timestamps
try:
    transcribe(WHISPER_BASE, long_clip)
except Exception as exc:
    print("(a) naive  ->", type(exc).__name__, ":", str(exc).splitlines()[0][:130])

# (b) chunked: 30 s windows with overlap, batched
out_c, secs_c = transcribe(WHISPER_BASE, long_clip, chunk_length_s=30, batch_size=8,
                           max_new_tokens=256, ignore_warning=True)
print(f"\n(b) chunked ({secs_c:.1f}s): {out_c['text'].strip()[:150]}…")

# (c) sequential long-form: Whisper predicts its own timestamps and slides the window
out_t, secs_t = transcribe(WHISPER_BASE, long_clip, return_timestamps=True)
print(f"\n(c) sequential ({secs_t:.1f}s), {len(out_t['chunks'])} segments:")
for ch in out_t["chunks"][:6]:
    s, e = ch["timestamp"]
    print(f"    [{s:6.2f} → {e if e is None else round(e,2)}]  {ch['text'].strip()[:60]}")

Chunking is faster because it batches, but it can cut a word at a boundary. Sequential decoding is
slower and keeps Whisper's own context across the file. Timestamps are what make subtitles, search
and diarisation possible.

## 5. Choosing a dataset

Four axes decide it: **hours**, **domain**, **speaking style** (narrated vs spontaneous), and
**transcription formatting** (casing and punctuation). If your labels are uppercase and unpunctuated,
your model will be too.

| Dataset | Hours | Domain | Style | Cased | Punctuated | License |
|---|---|---|---|---|---|---|
| LibriSpeech | 960 | Audiobook | Narrated | ❌ | ❌ | CC-BY-4.0 |
| Common Voice 11 | 3000 | Wikipedia | Narrated | ✅ | ✅ | CC0-1.0 |
| VoxPopuli | 540 | EU Parliament | Oratory | ❌ | ✅ | CC0 |
| TED-LIUM | 450 | TED talks | Oratory | ❌ | ❌ | CC-BY-NC-ND |
| GigaSpeech | 10000 | Audiobook, podcast, YouTube | Narrated + spontaneous | ❌ | ✅ | apache-2.0 |
| SPGISpeech | 5000 | Financial meetings | Oratory + spontaneous | ✅ | ✅ | User agreement |
| Earnings-22 | 119 | Financial meetings | Oratory + spontaneous | ✅ | ✅ | CC-BY-SA-4.0 |
| AMI | 100 | Meetings | Spontaneous | ✅ | ✅ | CC-BY-4.0 |

- **narrated**: "Consider the task of training a model on a speech recognition dataset"
- **spontaneous**: "Let's uhh let's take a look at how you'd go about training a model on uhm a sp- speech recognition dataset"

The **ESB benchmark** ([arXiv 2210.13352](https://arxiv.org/abs/2210.13352)) exists because a model
that wins on LibriSpeech often loses everywhere else.

The course's fine-tuning chapter uses Common Voice 13 Dhivehi, which is a **gated** dataset (accept
its terms and log in first). This notebook avoids gated data.

## 6. Word Error Rate, by hand

$$\mathrm{WER} = \frac{S + I + D}{N}$$

Substitutions, insertions and deletions over the number of **reference** words. Lower is better, and
because the denominator is the reference length there is **no upper bound**.

In [ ]:
# Levenshtein DP + backtrace -> (ops, counts). ops are (kind, ref_word, hyp_word).
def edit_ops(ref, hyp):
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): d[i][0] = i
    for j in range(m + 1): d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + cost)
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        cost = 0 if (i > 0 and j > 0 and ref[i-1] == hyp[j-1]) else 1
        if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + cost:
            ops.append(("ok" if cost == 0 else "sub", ref[i-1], hyp[j-1])); i, j = i-1, j-1
        elif i > 0 and d[i][j] == d[i-1][j] + 1:
            ops.append(("del", ref[i-1], None)); i -= 1
        else:
            ops.append(("ins", None, hyp[j-1])); j -= 1
    ops.reverse()
    counts = {k: sum(1 for o, _, _ in ops if o == k) for k in ("sub", "ins", "del")}
    counts["N"] = n
    return ops, counts

r, h = "the cat sat on the mat", "the cat sit on the"
ops, c = edit_ops(r.split(), h.split())
for op, a, b in ops:
    print(f"  {op.upper():<4} {str(a or '-'):<8} → {b or '-'}")
wer = (c["sub"] + c["ins"] + c["del"]) / c["N"]
print(f"\nS={c['sub']} I={c['ins']} D={c['del']} N={c['N']}  →  WER = {wer:.3f}")

import jiwer
print("jiwer agrees:", round(jiwer.wer(r, h), 3))

Two traps beginners hit:

In [ ]:
# 1. WER is unbounded above, so "word accuracy = 1 - WER" can go negative.
chatty = "the cat sat on the mat and then it got up and left the room entirely"
_, c2 = edit_ops(r.split(), chatty.split())
w2 = (c2["sub"] + c2["ins"] + c2["del"]) / c2["N"]
print(f"rambling prediction: WER = {w2:.3f}, 'accuracy' = {1-w2:.3f}")

# 2. Corpus WER sums the errors and the reference words, THEN divides once.
#    It is NOT the mean of per-utterance WERs.
refs, hyps = [r, "hello world"], [h, "goodbye world"]
tot = {"sub": 0, "ins": 0, "del": 0, "N": 0}
per = []
for a, b in zip(refs, hyps):
    _, cc = edit_ops(a.split(), b.split())
    per.append((cc["sub"]+cc["ins"]+cc["del"]) / cc["N"])
    for k in tot: tot[k] += cc[k]
print(f"mean of per-utterance : {np.mean(per):.3f}")
print(f"corpus WER (correct)  : {(tot['sub']+tot['ins']+tot['del'])/tot['N']:.3f}")

# CER does the same over characters - kinder to near misses.
for a, b in (("similes", "similarly"), ("christmas", "christmanus")):
    _, cc = edit_ops(list(a), list(b))
    print(f"{a:<10} → {b:<12} CER {(cc['sub']+cc['ins']+cc['del'])/cc['N']:.3f}")

## 7. Orthographic vs normalised WER

Whisper writes `"Mr. Quilter's $20 isn't cheap."`; LibriSpeech's reference says
`"MISTER QUILTER'S TWENTY DOLLARS ISN'T CHEAP"`. Without normalisation you are scoring **formatting**,
not speech recognition.

- `BasicTextNormalizer` — lowercase, strip punctuation. Safe for any language.
- `EnglishTextNormalizer` — also expands contractions and numbers, and applies a 1740-entry
  British→American spelling map.

In [ ]:
from transformers import WhisperTokenizer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer, EnglishTextNormalizer

# NOTE: the course writes `from transformers import BasicTextNormalizer`, which no longer works.
tok = WhisperTokenizer.from_pretrained(WHISPER_BASE)
basic = BasicTextNormalizer()
english = EnglishTextNormalizer(getattr(tok, "english_spelling_normalizer", None) or {})

probe = "Mr. Quilter's 20 dollars isn't cheap."
print("raw     :", repr(probe))
print("basic   :", repr(basic(probe)))
print("english :", repr(english(probe)))

In [ ]:
def corpus_wer(refs, hyps):
    tot = {"sub": 0, "ins": 0, "del": 0, "N": 0}
    for a, b in zip(refs, hyps):
        _, cc = edit_ops(a.split(), b.split())
        for k in tot: tot[k] += cc[k]
    return (tot["sub"] + tot["ins"] + tot["del"]) / max(tot["N"], 1)

N_EVAL = 8
refs  = [ds[i]["text"] for i in range(N_EVAL)]
audio = [ds[i]["audio"]["array"] for i in range(N_EVAL)]
secs_audio = sum(len(a) / SAMPLING_RATE for a in audio)

results = []
for model_id in (CTC_MODEL, WHISPER_TINY, WHISPER_BASE, WHISPER_SMALL):
    hyps, elapsed = [], 0.0
    for a in audio:
        out, s = transcribe(model_id, a); hyps.append(out["text"]); elapsed += s
    ortho = corpus_wer(refs, [x.strip() for x in hyps])
    nr = [english(x) for x in refs]; nh = [english(x) for x in hyps]
    keep = [i for i in range(len(nr)) if nr[i].strip()]
    norm = corpus_wer([nr[i] for i in keep], [nh[i] for i in keep])
    results.append((model_id.split("/")[-1], ortho, norm, secs_audio / elapsed))
    print(f"{model_id:<28} ortho {ortho:.3f}   norm {norm:.3f}   RTFx {secs_audio/elapsed:.1f}x")

In [ ]:
labels = [x[0] for x in results]
x = np.arange(len(labels))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.bar(x - 0.2, [r[1] for r in results], 0.4, label="orthographic", color="tab:red")
ax1.bar(x + 0.2, [r[2] for r in results], 0.4, label="normalised", color="tab:green")
ax1.set_xticks(x, labels, rotation=15, ha="right"); ax1.set_ylabel("WER"); ax1.legend()
ax1.set_title("Normalisation forgives casing and punctuation")
ax2.scatter([r[3] for r in results], [r[2] for r in results], s=90)
for r_ in results:
    ax2.annotate(r_[0], (r_[3], r_[2]), textcoords="offset points", xytext=(6, 5), fontsize=8)
ax2.set(xlabel="RTFx (higher = faster)", ylabel="normalised WER", title="Bottom-right is what you want")
plt.tight_layout(); plt.show()

Whisper's **orthographic** WER lands near 1.0 — not because it got every word wrong, but because
LibriSpeech's references are UPPERCASE AND UNPUNCTUATED, so `He` vs `HE` counts as a substitution.
Wav2Vec2 looks better orthographically only because it happens to *shout* in the same format.

The lesson is not "always normalise". It is that orthographic WER only means something when the
reference is formatted the way you want output — which is exactly why ESB argues for orthographic WER
**and** for datasets whose labels carry real casing and punctuation.

## 8. What fine-tuning actually feeds the model

In [ ]:
from dataclasses import dataclass
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(WHISPER_TINY, language="english", task="transcribe")
print("advertised prefix:", processor.tokenizer.convert_ids_to_tokens(processor.tokenizer.prefix_tokens))

# A REAL BUG: on transformers 4.57 the *fast* tokenizer ignores the language/task given to
# from_pretrained() when it encodes. Labels come out missing <|en|><|transcribe|>, while
# generation DOES force them - so you would train on one prompt format and decode with another.
before = processor.tokenizer("hello world")["input_ids"]
processor.tokenizer.set_prefix_tokens(language="english", task="transcribe")
after = processor.tokenizer("hello world")["input_ids"]
print("encoded before   :", processor.tokenizer.convert_ids_to_tokens(before)[:4])
print("after the fix    :", processor.tokenizer.convert_ids_to_tokens(after)[:4])

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]
    out = processor(audio=audio["array"], sampling_rate=audio["sampling_rate"], text=example["text"])
    # Do NOT write out["labels"][0]: for a single string the tokenizer returns a FLAT
    # list of ids, so [0] is one integer and the collator's pad() would explode.
    out["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    return out

rows = [prepare_dataset(ds[i]) for i in (0, 1)]
for k, row in enumerate(rows):
    print(f"[{k}] input_features {np.asarray(row['input_features']).shape}   "
          f"labels {len(row['labels'])} tokens   {row['input_length']:.1f}s")

Note the asymmetry: every clip becomes the **same** `(80, 3000)` block because Whisper pads to 30
seconds, so the audio side needs no padding logic. The labels are ragged, so they do. **That is the
entire reason the data collator exists.**

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: object
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"][0]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # -100 is torch cross_entropy's ignore_index: padded positions contribute no loss.
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        # The model re-adds the start token when it shifts labels right, so drop ours.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

model = WhisperForConditionalGeneration.from_pretrained(WHISPER_TINY)
collator = DataCollatorSpeechSeq2SeqWithPadding(processor, model.config.decoder_start_token_id)

padded = processor.tokenizer.pad([{"input_ids": r["labels"]} for r in rows], return_tensors="pt")
batch = collator(rows)
short = int(np.argmin([len(r["labels"]) for r in rows]))
print("input_features:", tuple(batch["input_features"].shape))
print("labels        :", tuple(padded["input_ids"].shape), "->", tuple(batch["labels"].shape))
print(f"\nrow {short} (the shorter one) tail:")
print("  before:", padded["input_ids"][short][-8:].tolist())
print("  after :", batch["labels"][short][-8:].tolist())
print(f"\nstart token {model.config.decoder_start_token_id} stripped; labels[:,0] is now",
      batch["labels"][:, 0].tolist())

## Fine-tuning (the hands-on recipe)

The cell below is the real training loop, gated behind a flag so this notebook stays fast. It fine-tunes
`whisper-tiny` on MINDS-14 en-US, first 450 examples for training and the rest for evaluation, and
passes when the **normalised WER drops below 0.37** (a *fraction*, not a percentage).

On CPU this takes hours, so run it on a GPU. For the graded exercise use
[`colab_handson.ipynb`](colab_handson.ipynb), which is the same recipe with a Hub push and full
explanations. Locally, `finetune.py` gives you a one-minute CPU smoke test of the same pipeline.

In [ ]:
RUN_TRAINING = False  # set True on a GPU/Colab box (and `uv sync --extra training`)

In [ ]:
if RUN_TRAINING:
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
    import evaluate
    from datasets import DatasetDict

    minds = load_minds14("en-US")
    minds = minds.select_columns(["audio", "transcription"]).rename_column("transcription", "sentence")
    minds = minds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
    n_train = min(450, int(len(minds) * 0.8))
    data = DatasetDict(train=minds.select(range(n_train)),
                       test=minds.select(range(n_train, len(minds))))

    def prep(example):
        a = example["audio"]
        out = processor(audio=a["array"], sampling_rate=a["sampling_rate"], text=example["sentence"])
        out["input_length"] = len(a["array"]) / a["sampling_rate"]
        return out

    data = data.map(prep, remove_columns=data.column_names["train"], num_proc=1)
    data = data.filter(lambda l: l < 30.0, input_columns=["input_length"]).remove_columns(["input_length"])

    metric = evaluate.load("wer")
    def compute_metrics(pred):
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str = processor.batch_decode(pred.predictions, skip_special_tokens=True)
        label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
        wer_ortho = metric.compute(predictions=pred_str, references=label_str)
        pn = [basic(p) for p in pred_str]; ln = [basic(l) for l in label_str]
        keep = [i for i in range(len(ln)) if ln[i]]
        wer = metric.compute(predictions=[pn[i] for i in keep], references=[ln[i] for i in keep])
        return {"wer_ortho": wer_ortho, "wer": wer}     # fractions, not percentages

    model.config.use_cache = False
    model.generation_config.use_cache = True
    model.generation_config.language = "english"
    model.generation_config.task = "transcribe"

    args = Seq2SeqTrainingArguments(
        output_dir="whisper-tiny-finetuned-minds14-en",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        learning_rate=1e-5, lr_scheduler_type="constant_with_warmup", warmup_steps=50,
        max_steps=600, gradient_checkpointing=True, fp16=True, fp16_full_eval=True,
        eval_strategy="steps", save_strategy="steps",   # NOT evaluation_strategy
        eval_steps=50, save_steps=50, save_total_limit=2,
        predict_with_generate=True, generation_max_length=225, logging_steps=25,
        load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
        push_to_hub=False, report_to=["none"],
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=data["train"], eval_dataset=data["test"],
        data_collator=collator, compute_metrics=compute_metrics,
        processing_class=processor,                     # NOT tokenizer=
    )
    trainer.train()
    m = trainer.evaluate()
    print(m)
    print("PASS" if m["eval_wer"] < 0.37 else "NOT YET")
else:
    print("skipped (set RUN_TRAINING = True on a GPU box, or use colab_handson.ipynb)")

---

### The hands-on

Fine-tune `openai/whisper-tiny` on `PolyAI/minds14` **en-US**, first 450 examples for training and the
rest for evaluation, `num_proc=1` when mapping, and push to the Hub with `dataset_tags`,
`finetuned_from` and `tasks`. You pass at **normalised WER < 0.37** — reported as a fraction.
Open [`colab_handson.ipynb`](colab_handson.ipynb) on a T4 runtime and run all cells.

🎉 That's Unit 5. Next: Unit 6, from text to speech.